In [1]:
import duckdb
import os
cn = duckdb.connect()

In [2]:
sql = f"""
    create or replace secret gcs_secrets (
        TYPE GCS
        ,KEY_ID '{os.getenv("GCS_KEY")}'
        ,SECRET '{os.getenv("GCS_SECRET")}'
    )
"""
cn.execute(sql)

In [3]:
sql = f"""
ATTACH OR REPLACE 'ducklake:gcs_wh.ducklake' as gcs_wh (DATA_PATH 'gs://{os.getenv("GCS_BUCKET")}/ducklake');
"""
cn.execute(sql)

In [4]:
sql = """
    create or replace table gcs_wh.orders as
    select 1 as order_id, '2024-01-01'::date as order_date, 100.0 as amount
    union all
    select 2 as order_id, '2024-01-02'::date as order_date, 150.0 as amount
"""
cn.execute(sql)

In [5]:
cn.sql("select * from gcs_wh.orders").show()

┌──────────┬────────────┬──────────────┐
│ order_id │ order_date │    amount    │
│  int32   │    date    │ decimal(4,1) │
├──────────┼────────────┼──────────────┤
│        1 │ 2024-01-01 │        100.0 │
│        2 │ 2024-01-02 │        150.0 │
└──────────┴────────────┴──────────────┘

